<a href="https://colab.research.google.com/github/trainocate-japan/developing-agentic-ai-with-langchain/blob/main/chap02/exercise/starter/chap02_exercise_2B_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 演習 2-B: Function Calling 手動 1 周 — ヘルプデスク Step 1

**研修コース「Agentic AI 開発実践 - LangChain 版」/ 第2章「LLM API の基礎」**

この Notebook は演習 2-B の**問題 (starter)** です。コード中の **`# TODO`** の箇所を、
ヒントを頼りに自分で埋めて完成させてください。TODO 以外のセルは完成済みなので、
**3 つの TODO (①②③) に集中**できます。

ハンズオン 2-A の後半 (セクション 7) で、`get_weather` を題材に Function Calling の手動 1 周を
**動かして確認**しました。この演習では、その**同じ 4 ステップ**を、今度は helpdesk の
`get_system_status` ツールで**自分の手で**組み立てます。
詰まったら、各 TODO の直前の Markdown ヒントを読み返しましょう。
それでも解けないときだけ、`solution` Notebook で答え合わせをしてください。

## やること — Function Calling の 4 ステップ

社内 IT ヘルプデスクの稼働状況に答える `get_system_status(service)` ツールを使い、
Function Calling のループを**手動で 1 周**させます。これは本コースの演習ストーリー
「**社内 IT ヘルプデスクエージェント**」の第一歩 (Step 1) です。

| ステップ | 内容 | あなたが埋める TODO |
|---|---|---|
| ① ツール定義 | `tools` に関数の仕様を JSON Schema で書く | **TODO①** (`parameters`) |
| ② tool_calls 受信 | 呼び出し宣言を取り出し `json.loads` する | **TODO②** |
| ③ アプリ側で実行 | 関数を実行する | (完成済み) |
| ④ 結果を返す | assistant + tool メッセージを履歴に積む | **TODO③** |

## 前提条件

- **ハンズオン 2-A を完了している**こと (同一環境で続けて実施。後半の `get_weather` デモで FC の 4 ステップを確認済み)
- Google Colab で開き、Colab シークレットに `OPENAI_API_KEY` を登録済みであること

## 完成の目安

- 「勤怠システムは動いていますか?」への最終応答が得られる
- 「こんにちは」では `tool_calls` が返らない (モデルがツール不要と正しく判断する)

## 所要時間

約 18 分

---
> **モデル名について**: モデル名は変数 `MODEL` に集約しています (例: `MODEL = "gpt-5.4"`)。
> 研修実施時は講師が指定する最新モデル名に差し替えてください。


## 0. セットアップ (このセクションは完成済み・実行するだけ)

ハンズオン 2-A と同じセットアップです。すでに同一セッションで 2-A を実行済みなら流すだけで構いません。


In [ ]:
# openai パッケージを最新版へ (研修実施時は最新版にピン留め推奨)
!pip install -U openai

In [ ]:
import os
import json

# Colab シークレットから API キーを読み込む。Colab 以外では環境変数をそのまま使う
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass  # Colab 以外では環境変数 OPENAI_API_KEY が設定済みとみなす

from openai import OpenAI

client = OpenAI()          # 環境変数 OPENAI_API_KEY からキーを自動取得
MODEL = "gpt-5.4"          # モデル名は変数に集約 (研修実施時に最新へ差し替え)

print("準備完了。使用モデル:", MODEL)

---

## 1. 題材の関数 (このセクションは完成済み・実行するだけ)

モデルに「呼び出させる」ための関数です。固定の稼働状況を返す**ダミー実装**です
(本物の監視 API は使いません)。ここは手を加えなくて構いません。


In [ ]:
# 社内システムの稼働状況を返すダミー関数 (完成済み)
SYSTEM_STATUS = {
    "勤怠システム": "正常稼働中",
    "経費精算システム": "正常稼働中",
    "メールサーバー": "一部遅延あり (調査中)",
    "VPN": "メンテナンス中 (本日 22:00 まで)",
}


def get_system_status(service: str) -> str:
    """指定された社内システムの現在の稼働状況を返す (ダミー実装)"""
    status = SYSTEM_STATUS.get(service, "不明 (登録されていないシステムです)")
    return f"{service}の稼働状況: {status}"


# 動作確認
print(get_system_status("勤怠システム"))

---

## 2. ステップ①: ツールを定義してリクエストする 【TODO①】

関数の存在をモデルに伝える**ツール定義**を書きます。
`parameters` の部分は **JSON Schema** という標準形式で引数の仕様を記述します。

### TODO① のヒント

下のセルの `parameters` を埋めてください。
- `service` という**文字列引数を必須**で定義します
- 埋めるキーは `type` / `properties` / `required` の 3 つ:
  - `"type": "object"` (引数全体は 1 つのオブジェクト)
  - `"properties"`: `service` を `{"type": "string", "description": "..."}` で定義
  - `"required"`: `["service"]` (必須引数のリスト)
- `description` はモデルがツールを選ぶ唯一の判断材料です。**丁寧に**書きましょう
  (例: 「稼働状況を知りたい社内システムの名称。例: 勤怠システム」)

> 形が分からなくなったら、ハンズオン 2-A の `get_weather` のツール定義が雛形です。
> `parameters` は `type` (= `"object"`) / `properties` (各引数の `type` と `description`) / `required` (必須引数名のリスト)
> を持つ JSON Schema でした。


In [ ]:
# ツール定義: get_system_status の「取扱説明書」を JSON で書く
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_system_status",
            # description はモデルがツールを選ぶ唯一の判断材料。丁寧に書く
            "description": "社内システム (勤怠システム、経費精算システム、メールサーバー、VPN 等) の現在の稼働状況を取得する",
            # ▼▼▼ TODO①: parameters を JSON Schema で埋める ▼▼▼
            "parameters": {
                # TODO: "type" を "object" にする
                # TODO: "properties" に service (type=string, description 付き) を定義する
                # TODO: "required" に必須引数 service を指定する
            },
            # ▲▲▲ TODO① ここまで ▲▲▲
        },
    }
]

print("ツール定義:", tools[0]["function"]["name"])

定義ができたら、`tools` を添えて「**勤怠システムは動いていますか?**」と送ります (このセルは完成済み)。

**期待される結果**: TODO①が正しければ、モデルは直接答えず `get_system_status` の呼び出し宣言を返します。


In [ ]:
# 「勤怠システムは動いていますか?」を tools 付きで送信する (完成済み)
messages = [{"role": "user", "content": "勤怠システムは動いていますか?"}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,          # ツール定義を添える
)

print("送信完了。次のステップでレスポンスを確認します。")

---

## 3. ステップ②: tool_calls を受け取る 【TODO②】

モデルがツールを必要と判断すると、`finish_reason` が `"tool_calls"` になり、
`message.tool_calls` に呼び出し宣言が入ります。

### TODO② のヒント

下のセルで、呼び出し宣言から**関数名**と**引数**を取り出してください。
- `tool_call = response.choices[0].message.tool_calls[0]` で最初の宣言を取得 (完成済み)
- `tool_call.function.name` … 関数名 (文字列)
- `tool_call.function.arguments` … 引数。**ここが要注意: dict ではなく JSON 文字列です**
- **`arguments` は `json.loads()` でパース**して dict にしてください
  (`tool_call.function.arguments["service"]` と書くと TypeError になります)


In [ ]:
# finish_reason の確認 (完成済み)
print("finish_reason:", response.choices[0].finish_reason)   # => "tool_calls" のはず

# 最初の呼び出し宣言を取得 (完成済み)
tool_call = response.choices[0].message.tool_calls[0]

# ▼▼▼ TODO②: 関数名と引数を取り出す ▼▼▼
# TODO: function_name に tool_call の関数名 (tool_call.function.name) を代入する
function_name = None
# TODO: arguments は JSON「文字列」。json.loads() で dict にパースして args に代入する
#       (ヒント: tool_call.function.arguments をパースする)
args = None
# ▲▲▲ TODO② ここまで ▲▲▲

print("function_name:", function_name)   # => "get_system_status"
print("args (dict)  :", args)            # => {"service": "勤怠システム"}

---

## 4. ステップ③: アプリ側で関数を実行する (このセルは完成済み)

宣言を読み取ったら、実行するのは**アプリ側の仕事**です。
TODO②で取り出した `args` を使って関数を呼びます。
この行が「**LLM は関数を実行しない**(実行するのはあなたのコード)」の証拠です。


In [ ]:
# アプリ側で関数を実行する (完成済み。TODO②の args を使う)
result = get_system_status(**args)
print("関数の実行結果:", result)   # => "勤怠システムの稼働状況: 正常稼働中"


---

## 5. ステップ④: 結果を返して最終応答を得る 【TODO③】

実行結果をモデルに伝えて最終応答をもらいます。
履歴には **2 つ**のメッセージを**正しい順序**で積みます。

### TODO③ のヒント

下のセルで、履歴 `messages` に 2 つのメッセージを追加してください。
1. **(a) 先に** `tool_calls` 入りの assistant メッセージを積む
   - これは `response.choices[0].message` (= モデルが返したメッセージ) **そのもの**です
   - `messages.append(response.choices[0].message)`
2. **(b) 次に** 実行結果を `role:"tool"` で積む。`tool_call_id` を一致させる
   - `{"role": "tool", "tool_call_id": ???, "content": result}`
   - **`tool_call_id` はモデルが発行した `tool_calls[0].id` をそのまま使う** (= `tool_call.id`)

> **順序が大事**: (a) を忘れたり (b) を先に積むと
> `BadRequestError: messages with role 'tool' must be a response to a preceding message
> with 'tool_calls'` になります。**「宣言→結果」のペア**で積みましょう。


In [ ]:
# ▼▼▼ TODO③: assistant メッセージ (tool_calls 入り) と tool メッセージを履歴に積む ▼▼▼
# TODO (a): tool_calls 入りの assistant メッセージ (response.choices[0].message) を先に append する

# TODO (b): 実行結果を role:"tool" で append する。
#           tool_call_id にはモデルが発行した id (tool_call.id) をそのまま入れ、content に result を入れる

# ▲▲▲ TODO③ ここまで ▲▲▲

# 全履歴を再送信 → モデルが結果を踏まえた最終応答を生成 (完成済み)
final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
print("最終応答:", final.choices[0].message.content)

ここまでで Function Calling の 1 周が完成します。
「勤怠システムは正常に稼働しています」のような最終応答が返れば成功です 🎉


---

## 6. tool_calls が返らないことの確認 (このセルは完成済み)

ツールが不要な質問では `tool_calls` が返らないことを確認します。
**期待される結果**: `finish_reason` が `"stop"`、`tool_calls` が `None`、`content` に挨拶の返事。


In [ ]:
# ツール不要の質問では tool_calls が返らないことを確認する (完成済み)
greeting = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "こんにちは"}],
    tools=tools,
)

print("finish_reason:", greeting.choices[0].finish_reason)        # => "stop"
print("tool_calls   :", greeting.choices[0].message.tool_calls)   # => None
print("content      :", greeting.choices[0].message.content)

---

## 7. (発展課題) while ループ化 + ツール 2 つ 【早く終わった人向け】

> ここは**早く終わった人向けの発展課題**です。必須ではありません。
> 基本課題 (1〜6) が動いてから取り組んでください。

ここまではツール呼び出しが 1 回でした。**ステップ 2〜4 を `finish_reason` が `"tool_calls"`
でなくなるまで while ループで回す**と、ツールの連鎖呼び出しに対応できます。
ツールも 2 つに増やします (`get_maintenance_schedule` を追加)。

**この while ループこそがエージェントの正体です。**
第3章の `create_agent` が、このループを自動化してくれます。

まず 2 つ目のツールとレジストリを用意します (この部分は完成済み)。


In [ ]:
# 2 つ目のダミーツール: メンテナンス予定を返す (完成済み)
MAINTENANCE_SCHEDULE = {
    "勤怠システム": "予定なし",
    "経費精算システム": "今週末 (土) 02:00-04:00 に定期メンテナンス",
    "メールサーバー": "予定なし",
    "VPN": "本日 20:00-22:00 にメンテナンス中",
}


def get_maintenance_schedule(service: str) -> str:
    """指定された社内システムのメンテナンス予定を返す (ダミー実装)"""
    schedule = MAINTENANCE_SCHEDULE.get(service, "不明 (登録されていないシステムです)")
    return f"{service}のメンテナンス予定: {schedule}"


# 関数名から実体を引くためのレジストリ (完成済み)
AVAILABLE_FUNCTIONS = {
    "get_system_status": get_system_status,
    "get_maintenance_schedule": get_maintenance_schedule,
}

# 2 ツール分の定義 (完成済み。TODO① が解ければこの形も読めるはず)
tools_multi = [
    {
        "type": "function",
        "function": {
            "name": "get_system_status",
            "description": "社内システムの現在の稼働状況を取得する",
            "parameters": {
                "type": "object",
                "properties": {"service": {"type": "string", "description": "システム名。例: 勤怠システム"}},
                "required": ["service"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_maintenance_schedule",
            "description": "社内システムのメンテナンス予定 (停止時間など) を取得する",
            "parameters": {
                "type": "object",
                "properties": {"service": {"type": "string", "description": "システム名。例: VPN"}},
                "required": ["service"],
            },
        },
    },
]

print("2 つのツール:", list(AVAILABLE_FUNCTIONS.keys()))

### 発展 TODO: while ループ本体を完成させる

下のループの TODO を埋めてください。ヒント:
- `finish_reason != "tool_calls"` なら最終応答が出たので `break`
- `tool_calls` は**リスト**。`for tool_call in msg.tool_calls:` で 1 つずつ処理する
  (parallel tool calls = 複数同時呼び出しに対応するため)
- 各 `tool_call` について: 関数名から `AVAILABLE_FUNCTIONS` で実体を引き、`json.loads` で引数をパースして実行
- 基本課題の TODO②③ とまったく同じ要領です (assistant を積む → tool を積む)


In [ ]:
# while ループ版の Function Calling (発展課題)
messages = [
    {"role": "system", "content": "あなたは社内 IT ヘルプデスクの一次対応担当です。必要に応じてツールで状況を調べて答えてください。"},
    {"role": "user", "content": "VPN は今使えますか? メンテナンス予定もあわせて教えてください。"},
]

max_turns = 5   # 無限ループ防止の安全弁
for turn in range(max_turns):
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools_multi)
    msg = response.choices[0].message

    # ▼▼▼ 発展 TODO ▼▼▼
    # TODO: finish_reason が "tool_calls" でなければ最終応答を表示して break する
    #       (response.choices[0].finish_reason を見る)

    # TODO (a): tool_calls 入りの assistant メッセージ (msg) を messages に積む

    # TODO: msg.tool_calls をループし、各 tool_call について…
    #   - AVAILABLE_FUNCTIONS[tool_call.function.name] で関数の実体を取得
    #   - json.loads(tool_call.function.arguments) で引数を dict にする
    #   - 関数を実行して result を得る
    #   - (b) {"role": "tool", "tool_call_id": tool_call.id, "content": result} を messages に積む
    # ▲▲▲ 発展 TODO ここまで ▲▲▲
    pass
else:
    print("[警告] 最大ターン数に達しました。")

---

## まとめ — 提出前のチェックリスト

- [ ] **TODO①**: ツール定義の `parameters` (JSON Schema) を埋めた → ②で `tool_calls` が返った
- [ ] **TODO②**: `tool_calls[0]` から関数名・引数を取り出し、`json.loads` で dict にした
- [ ] **TODO③**: assistant (tool_calls 入り) → tool (`tool_call_id` 一致) の順で履歴に積んだ
- [ ] 「勤怠システムは動いていますか?」への最終応答が得られた
- [ ] 「こんにちは」では `tool_calls` が返らなかった

### 定番のつまずき (もしエラーが出たら)

- `TypeError: string indices must be integers` → `arguments` を `json.loads` せず dict 扱いした (TODO②)
- `BadRequestError: messages with role 'tool' must be a response to a preceding message with 'tool_calls'`
  → assistant メッセージ (tool_calls 入り) を積み忘れ、または順序が逆 (TODO③)
- ツールが呼ばれない (`tool_calls` が None のまま) → `parameters` の JSON Schema が不完全 (TODO①)

### この完成コードは保存してください

**完成したコードは必ず保存しておいてください。** 第3章で `create_agent` 版と並べて diff を取り、
フレームワークが何を肩代わりしているかを、自分が書いたこのコードで確認します。
